# TeleClinic Platform — Month 3 Learning Review
**Data Analyst & MEL Associate | Take-Home Assessment**

**Candidate:** Herve Twahirwa  
Period: February – April 2026 (Weeks 1–13)  
Dataset: `TeleClinic_Candidate_Dataset_Month3.xlsx`

---

This notebook contains the **Month 3 learning review analysis** for the TeleClinic take-home assessment. Read the Data Dictionary tab before interpreting results.


## 0. Setup & Data Loading

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = Path("TeleClinic_Candidate_Dataset_Month3.xlsx")
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH.resolve()}"


In [ ]:
SHEET_MAP = {
    "patients": "👥 Patients",
    "consultations": "📅 Consultations",
    "followups": "🔁 Follow-Ups",
    "lab_tests": "🧪 Lab Tests",
    "prescriptions": "💊 Prescriptions",
    "referrals": "↗️ Referrals",
    "insurance_log": "🔐 Insurance Log",
}

tables = {key: pd.read_excel(DATA_PATH, sheet_name=sheet) for key, sheet in SHEET_MAP.items()}
patients = tables["patients"]
consultations = tables["consultations"]
followups = tables["followups"]
lab_tests = tables["lab_tests"]
prescriptions = tables["prescriptions"]
referrals = tables["referrals"]
insurance_log = tables["insurance_log"]

print("Row counts:")
for name, df in tables.items():
    print(f"  {name:18s} {len(df):>5,}")

In [ ]:
DATE_COLS = {
    "patients": ["registration_date"],
    "consultations": ["booked_datetime"],
    "followups": ["followup_booked_date"],
    "lab_tests": ["requested_datetime", "upload_datetime"],
    "prescriptions": ["created_datetime", "dispensed_datetime"],
    "referrals": ["initiated_datetime", "authorisation_datetime"],
    "insurance_log": ["attempt_datetime"],
}

for table_name, cols in DATE_COLS.items():
    df = tables[table_name]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

---
## Part 1: Data Quality Audit (20 pts)

**Deliverable:** List every data quality problem found. For each: *what*, *likely explanation*, *impact on reporting*.

Also address:
- Join problem across tables (~18% referrals noted in dictionary)
- District with implausibly low consultation volumes
- Most concerning patient safety issue

### 1.1 Missing values & duplicates (all tables)

In [ ]:
dq_frames = []
for name, df in tables.items():
    report = pd.DataFrame({
        "table": name,
        "column": df.columns,
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean() * 100).values,
        "dtype": df.dtypes.astype(str).values,
    }).sort_values("null_pct", ascending=False)
    dq_frames.append(report[report["null_pct"] > 0])

missing_summary = pd.concat(dq_frames, ignore_index=True)
missing_summary.sort_values(["table", "null_pct"], ascending=[True, False])

In [ ]:
pk_checks = [
    (patients, "patient_id", "patients"),
    (consultations, "consultation_id", "consultations"),
    (followups, "followup_id", "followups"),
    (lab_tests, "lab_id", "lab_tests"),
    (prescriptions, "prescription_id", "prescriptions"),
    (referrals, "referral_id", "referrals"),
    (insurance_log, "log_id", "insurance_log"),
]
pd.DataFrame([
    {"table": name, "id_col": col, "duplicate_ids": int(df[col].duplicated().sum())}
    for df, col, name in pk_checks
])

### 1.3 Referential integrity & referral data gaps

The data dictionary notes **~18% of referral records have a deliberate issue**. Run join checks across all tables, then inspect referral-specific gaps (e.g. missing `authorised` status where NULL means record missing).

In [ ]:
join_checks = [
    (consultations, patients, "patient_id", "patient_id", "consultations", "patients"),
    (followups, consultations, "original_consult_id", "consultation_id", "followups", "consultations"),
    (followups, patients, "patient_id", "patient_id", "followups", "patients"),
    (lab_tests, consultations, "consultation_id", "consultation_id", "lab_tests", "consultations"),
    (lab_tests, patients, "patient_id", "patient_id", "lab_tests", "patients"),
    (prescriptions, consultations, "consultation_id", "consultation_id", "prescriptions", "consultations"),
    (prescriptions, patients, "patient_id", "patient_id", "prescriptions", "patients"),
    (referrals, consultations, "consultation_id", "consultation_id", "referrals", "consultations"),
    (referrals, patients, "patient_id", "patient_id", "referrals", "patients"),
    (insurance_log, patients, "patient_id", "patient_id", "insurance_log", "patients"),
]

join_results = []
for child, parent, ck, pk, cn, pn in join_checks:
    parent_ids = set(parent[pk].dropna())
    orphan_rows = (~child[ck].isin(parent_ids)).sum()
    join_results.append({
        "child_table": cn,
        "parent_table": pn,
        "join_key": ck,
        "child_rows": len(child),
        "orphan_rows": int(orphan_rows),
        "orphan_pct": round(orphan_rows / len(child) * 100, 1) if len(child) else 0,
    })

join_summary = pd.DataFrame(join_results).sort_values("orphan_pct", ascending=False)
join_summary

In [ ]:
referral_missing_auth = referrals[referrals["authorised"].isna()]
print(f"Referrals with missing authorisation record: {len(referral_missing_auth)} ({len(referral_missing_auth)/len(referrals)*100:.1f}%)")
referrals["authorised"].value_counts(dropna=False)

### 1.4 Cross-table consistency checks

In [ ]:
# Patients registered but never consulted
patients_no_consult = set(patients["patient_id"]) - set(consultations["patient_id"])
print(f"Registered patients with zero consultations: {len(patients_no_consult)}")

# Consultations for patients not in registry
consult_no_patient = set(consultations["patient_id"]) - set(patients["patient_id"])
print(f"Consultations with unknown patient_id: {len(consult_no_patient)}")

# Incomplete registration but has consultations?
incomplete_with_consult = patients[
    (patients["registration_complete"] == "No") &
    (patients["patient_id"].isin(consultations["patient_id"]))
]
print(f"Incomplete registration but has consultation: {len(incomplete_with_consult)}")

In [ ]:
# Completed consultations should have diagnosis / notes / call_type populated
completed = consultations[consultations["status"] == "Completed"].copy()
print(f"Completed consultations: {len(completed)}")

for col in ["diagnosis_category", "notes_entered", "icd_code_entered", "call_type"]:
    null_n = completed[col].isna().sum()
    print(f"  {col} missing among completed: {null_n} ({null_n/len(completed)*100:.1f}%)")

# Notes / ICD compliance among completed
print(f"\nNotes entered (Yes) among completed: {(completed['notes_entered']=='Yes').mean()*100:.1f}%")
print(f"ICD code entered (Yes) among completed: {(completed['icd_code_entered']=='Yes').mean()*100:.1f}%")

In [ ]:
# Lab: result uploaded but no TAT / upload datetime
lab_uploaded = lab_tests[lab_tests["result_uploaded"] == "Yes"]
lab_missing_tat = lab_uploaded["tat_hours"].isna().sum()
lab_not_viewed = lab_uploaded[lab_uploaded["clinician_viewed"] != "Yes"]
print(f"Lab results uploaded: {len(lab_uploaded)}")
print(f"  Missing tat_hours despite upload: {lab_missing_tat}")
print(f"  Clinician has not viewed result: {len(lab_not_viewed)} ({len(lab_not_viewed)/len(lab_uploaded)*100:.1f}%)")

# Prescriptions: dispensed but missing lag
rx_dispensed = prescriptions[prescriptions["dispensed"] == "Yes"]
rx_missing_lag = rx_dispensed["lag_hours"].isna().sum()
print(f"\nPrescriptions dispensed: {len(rx_dispensed)}")
print(f"  Missing lag_hours despite dispensed: {rx_missing_lag}")

### 1.5 District with implausibly low consultation volume

Compare consultation counts by district against patient registrations. Three possible explanations + data needed to investigate.

In [ ]:
district_patients = patients.groupby("district").size().rename("registrations")
district_consults = consultations.groupby("district").size().rename("consultations")
district_volume = pd.concat([district_patients, district_consults], axis=1).fillna(0).astype(int)
district_volume["consult_per_reg"] = (
    district_volume["consultations"] / district_volume["registrations"]
).round(2)
district_volume.sort_values("consultations")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
district_volume.sort_values("consultations")["consultations"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Consultations by District (Month 3)")
ax.set_xlabel("District")
ax.set_ylabel("Consultation count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 1.6 Patient safety flags

Review: missing clinical documentation, unreviewed lab results, unauthorised referrals, etc.

In [ ]:
safety_flags = []

# Completed consults without notes
no_notes = completed[completed["notes_entered"] != "Yes"]
safety_flags.append({"issue": "Completed consult, no clinical notes", "count": len(no_notes)})

# Completed consults without ICD
no_icd = completed[completed["icd_code_entered"] != "Yes"]
safety_flags.append({"issue": "Completed consult, no ICD code", "count": len(no_icd)})

# Lab uploaded, clinician never viewed
safety_flags.append({"issue": "Lab result uploaded, clinician not viewed", "count": len(lab_not_viewed)})

# Referrals not authorised (incl. null)
ref_unauth = referrals[(referrals["authorised"] != "Yes") | referrals["authorised"].isna()]
safety_flags.append({"issue": "Referral not authorised or status missing", "count": len(ref_unauth)})

pd.DataFrame(safety_flags).sort_values("count", ascending=False)

### 1.7 Data Quality Issue Log

I read the data dictionary first, then checked missing values, duplicate IDs, and joins across all tables before running any performance numbers.

| # | Issue | Likely explanation | Impact on reporting |
|---|-------|--------------------|---------------------|
| 1 | 18 referrals (17.6%) with missing authorisation status | Referral started clinically but authorisation never recorded or did not sync (NULL = missing record) | RM metrics unreliable; processing time unknown for these cases |
| 2 | 122 completed consults without notes; 169 without ICD | Clinicians closing under time pressure; no hard stop at sign-off | Audit and continuity weakened; use completed consults only for quality metrics |
| 3 | 307 rows with null diagnosis/notes/ICD/call_type | Expected for cancelled/no-show — not a bug | Inflates gaps if mixed into quality denominators |
| 4 | 213 incomplete registrations / failed validation | Insurance API: NID not found, expired cover, timeouts | 491 registered patients never booked |
| 5 | 62 lab results uploaded but not viewed | No alert when results sit unreviewed | Patient safety — results may never reach patient |
| 6 | 23 lab tests without uploaded results | Lab capacity or sync delay | Lab completion/TAT incomplete |
| 7 | 81 prescriptions not dispensed | Non-collection, stock-out, pathway change | Continuity gap |
| 8 | 15 referrals not authorised + 18 unknown | Triage rejection or admin backlog | Patient may not know referral stalled |
| 9 | failure_reason null on 987 insurance rows | Only filled on failed attempts | Misleading if reported without filtering |

**Join problem — how found, handled, excluded:**  
I ran orphan checks on every foreign key. All IDs match (zero orphans). The referral issue is incomplete records: 18 referrals have valid consultation_id but NULL `authorised` and related fields. I report authorisation on known-status records only (69/84 = 82.1%) and flag the 18 unknowns separately — I did not drop them, because they are real initiated referrals.

**Low-volume district — Musanze (6 consults / 21 registrations, ratio 0.29):**  
1. Later district rollout — fewer active weeks in the 13-week window.  
2. Rural connectivity and awareness — all Musanze patients are rural-classified.  
3. Small sample — 21 registrations means a few no-shows move the ratio a lot.

**Additional data needed:** District activation dates, clinician roster hours, outreach logs, network coverage maps.

**Most concerning patient safety issue:**  
The 62 uploaded lab results clinicians never viewed. In telemedicine there is no ward round to catch this — the platform is the only link between lab and patient.

---
## Part 2: Metrics & Analysis (25 pts)

Identify the **5 most important metrics** for Month 3 platform performance. Define precisely, calculate, justify.

### 2.1 Derived datasets (use for consistent denominators)

In [ ]:
# Standard subsets
completed_consults = consultations[consultations["status"] == "Completed"].copy()
matched_referrals = referrals[referrals["consultation_id"].isin(consultations["consultation_id"])].copy()

print(f"Total consultations: {len(consultations):,}")
print(f"Completed: {len(completed_consults):,} ({len(completed_consults)/len(consultations)*100:.1f}%)")
print(f"No-Show: {(consultations['status']=='No-Show').sum():,}")
print(f"Cancelled: {(consultations['status']=='Cancelled').sum():,}")

### 2.2 Metric calculations *(customise — these are suggested starting metrics)*

In [ ]:
m1_num, m1_den = len(completed_consults), len(consultations)
m2_num = ((completed_consults["notes_entered"] == "Yes") & (completed_consults["icd_code_entered"] == "Yes")).sum()
m2_den = len(completed_consults)
m3_subset = lab_tests[lab_tests["result_uploaded"] == "Yes"]
m3_num, m3_den = (m3_subset["clinician_viewed"] == "Yes").sum(), len(m3_subset)
m4_num, m4_den = (prescriptions["dispensed"] == "Yes").sum(), len(prescriptions)
first_attempt = insurance_log[insurance_log["attempt_number"] == 1]
m5_num, m5_den = (first_attempt["success"] == "Yes").sum(), len(first_attempt)

metrics_df = pd.DataFrame([
    {"metric": "Consultation completion rate", "value": f"{m1_num/m1_den*100:.1f}%", "numerator": m1_num, "denominator": m1_den, "exclusions": "All booked consultations", "justification": "Core engagement / access funnel metric"},
    {"metric": "Full clinical documentation rate (notes + ICD)", "value": f"{m2_num/m2_den*100:.1f}%", "numerator": m2_num, "denominator": m2_den, "exclusions": "Completed consultations only", "justification": "Clinical quality / governance"},
    {"metric": "Lab result clinician review rate", "value": f"{m3_num/m3_den*100:.1f}%" if m3_den else "N/A", "numerator": m3_num, "denominator": m3_den, "exclusions": "Tests with uploaded results only", "justification": "Patient safety — results must reach clinician"},
    {"metric": "Prescription dispensing rate", "value": f"{m4_num/m4_den*100:.1f}%", "numerator": m4_num, "denominator": m4_den, "exclusions": "All prescriptions", "justification": "Care continuity — Rx must reach pharmacy"},
    {"metric": "First-attempt insurance validation success rate", "value": f"{m5_num/m5_den*100:.1f}%", "numerator": m5_num, "denominator": m5_den, "exclusions": "First validation attempt per log record", "justification": "Platform access barrier / onboarding friction"},
])
metrics_df

In [ ]:
# Supplementary: median TAT for labs and pharmacy lag
print(f"Median lab TAT (hours): {lab_tests['tat_hours'].median():.1f}")
print(f"Median pharmacy lag (hours): {prescriptions['lag_hours'].median():.1f}")
print(f"Median referral processing (hours): {matched_referrals['processing_hours'].median():.1f}")
print(f"Follow-up booking rate (of completed): {len(followups)/len(completed_consults)*100:.1f}%")

### 2.3 Metric selection notes

**Why these 5?**

| Metric | Result | Why it matters at Month 3 |
|--------|--------|---------------------------|
| Consultation completion rate | **72.4%** (807/1,114) | Does booked demand become delivered care? 14.9% no-show is wasted capacity |
| Full clinical documentation rate | **79.1%** (638/807) | Without notes + ICD, care is not auditable |
| Lab result clinician review rate | **82.3%** (288/350) | Uploaded but unreviewed = broken pathway and safety risk |
| Prescription dispensing rate | **84.5%** (440/521) | Remote prescribing only helps if pharmacy dispenses |
| First-attempt insurance validation | **82.2%** | Front-door barrier; Uninsured succeeds only 9.9% |

Each metric answers a different question — I did not want five versions of the same completion story.

**What we cannot assess from this data:**  
Clinical outcomes, patient satisfaction, referral arrival at facilities, clinical appropriateness of diagnoses/Rx, or whether notes are actually good (only that they exist).

**Single result that concerns me most:**  
About one in five care episodes has incomplete documentation or an unreviewed lab result. The 62 unreviewed results worry me most — a patient who tests and hears nothing may assume all is well. The platform is being used, but the results loop is not closed.

## Part 3: Equity Analysis (20 pts)

Rwanda benchmark: **~83% rural / 17% urban**.  
Assess equity across geography, channel, gender, insurance. Calculate **≥2 equity measures**.

In [ ]:
RWANDA_RURAL_PCT = 0.83

print("=== Patient registrations ===")
for col in ["urban_rural", "channel", "gender", "insurance_scheme"]:
    display((patients[col].value_counts(normalize=True) * 100).round(1).to_frame("pct"))

In [ ]:
print("=== Completed consultations ===")
for col in ["urban_rural", "channel"]:
    display((completed_consults[col].value_counts(normalize=True) * 100).round(1).to_frame("pct"))

In [ ]:
# Equity Measure 1: Rural Representation Index
# (share rural among users) / (national rural share)
rural_reg = (patients["urban_rural"] == "Rural").mean()
rural_consult = (completed_consults["urban_rural"] == "Rural").mean()

rri_reg = rural_reg / RWANDA_RURAL_PCT
rri_consult = rural_consult / RWANDA_RURAL_PCT

print(f"Rural share — registrations: {rural_reg*100:.1f}% (RRI={rri_reg:.2f})")
print(f"Rural share — completed consults: {rural_consult*100:.1f}% (RRI={rri_consult:.2f})")
print("RRI=1.0 → proportional; <1 under-represented; >1 over-represented")

In [ ]:
# Equity Measure 2: Coverage ratio by district (consultations / registrations)
district_equity = district_volume.copy()
district_equity["coverage_ratio"] = (
    district_equity["consultations"] / district_equity["registrations"]
).round(2)

# Merge urban/rural from patients
district_ur = patients.groupby("district")["urban_rural"].agg(lambda x: x.mode().iloc[0] if len(x) else "Unknown")
district_equity = district_equity.join(district_ur.rename("predominant_settlement"))
district_equity.sort_values("coverage_ratio")

In [ ]:
# Channel equity: USSD vs IremboApp by urban/rural
channel_equity = pd.crosstab(
    patients["urban_rural"], patients["channel"], normalize="index"
).mul(100).round(1)
channel_equity

In [ ]:
# Insurance equity: validation success by scheme
ins_equity = insurance_log.groupby("insurance_scheme").agg(
    attempts=("log_id", "count"),
    success_rate=("success", lambda s: (s == "Yes").mean()),
).assign(success_rate=lambda d: (d["success_rate"] * 100).round(1))
ins_equity.sort_values("success_rate")

### 3.1 Gates Foundation equity paragraph

After three months, TeleClinic registered 1,200 patients and completed 807 consultations across 12 districts — people are using it. Rural users make up 60% of activity against roughly 83% of Rwanda's population (RRI = 0.72), so we are not yet reaching rural communities proportionally. USSD carries about 41% of rural consults, which is genuinely important for feature-phone users, but urban districts — especially Gasabo — still dominate early volume, likely through IremboApp. Gender is balanced. The equity gap I would name honestly is insurance: uninsured patients succeed on validation less than 10% of the time, versus 78–90% on formal schemes. District coverage is uneven (Musanze 0.29 consult-to-registration ratio vs Rwamagana 1.09). **Recommended action:** USSD-focused outreach in the five lowest-coverage districts, plus a pilot 72-hour provisional access path for Uninsured patients while manual verification completes, with RRI tracked monthly toward 0.85 by Month 6.

### 3.2 Equity dimension this dataset cannot measure

**Socioeconomic vulnerability.** We have insurance scheme but not income or ability to pay. Expired CBHI (110 validation failures) may mean someone never gets to a consult — invisible here. I would add optional household quintile at registration (from DHS/EICV strata) and track validation failure → consult conversion by quintile.

---
## Part 4: Dashboard Design (20 pts)

**Audience:** Clinical Governance Committee  
**Central question:** *Is the platform delivering safe, quality care — and for whom?*

Interactive React dashboard: `teleclinic-dashboard/` (screenshot in submission PDF).  
Layout: **Safety | Clinical Quality | For Whom** on one screen.

Run with `cd teleclinic-dashboard && npm install && npm run dev`.

---
## Part 5: Learning Review Brief (15 pts)

### Platform Performance at Month 3
- **People are using it.** 1,200 registrations, 1,114 bookings, 807 completed (72.4%) in 13 weeks across 12 districts.  
- **Pathways partly close, safety gaps remain.** 84.5% of Rx dispensed; 62 uploaded lab results never reviewed by the ordering clinician.  
- **Rural reach below parity.** 60.1% rural users vs ~83% nationally; USSD helps but Musanze and others lag.

### What We Cannot Yet Conclude
- Whether telemedicine improves outcomes — no follow-up clinical or hospitalisation data.  
- Whether referred patients reach facilities — 18 missing authorisation records; no attendance data.  
- Whether the poorest benefit most — no socioeconomic or distance data.  
- Whether volume is sustainable — 13 weeks is too short to tell.

### One Recommended Change
Automated pathway closure tracker: flag unreviewed labs after 24h, undispensed Rx after 48h, un authorised referrals after 72h — escalate to clinical supervisor.

### Priority Question for Months 4–6
Does closing the lab and referral loop reduce care discontinuation among rural USSD users specifically?

---
## Reflection: North Star Metric (200–350 words)

**Metric:** Documented Care Completion Rate (DCCR) — share of all booked consultations that reach Completed with both notes and ICD recorded. **Month 3: 638 / 1,114 = 57.3%.**

Completion alone (72.4%) overstates success; documentation among completed (79.1%) hides bookings that never became care. DCCR combines both: did the patient get care, and was it auditable? Unlike lab or dispensing rates it applies to every consult; unlike RRI it measures delivery quality not reach.

Blind spots: it does not check clinical correctness, unreviewed labs (62), or undispensed Rx (81). It counts no-shows against the platform unfairly sometimes. It is equity-blind — DCCR could look fine while rural users stay under-represented (RRI 0.72). Still, 57.3% honestly describes Month 3: live and used, not yet reliably delivering governable care. I would track it monthly by district, channel, and settlement.